# Silver cleaning — Weather (combine stations and fill gaps)

This notebook combines the 18 per-station weather tables into one, parses the timestamp,
fills missing readings from nearby stations, and saves the result as
`hive_metastore.silver.silver_weather`.

The steps are:

1. **Combine** all station tables into one, tagging each row with its station.
2. **Parse the timestamp**.
3. **Fill gaps** — where a station is missing a reading, borrow the value from the nearest
   station that has one at the same time (trying the 3 closest in turn).
4. **Save**.

## 1. Combine the station tables

Load the common functions, then read all 18 station Delta tables, add a `location` column
to each, and union them into one table.

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/02_silver_cleaning/00_common_functions

In [0]:
# Path to weather_data location
base_path = "dbfs:/user/hive/warehouse/weather_data.db"

# List of table names (you can also automate this if needed)
locations = [
    "alagoa", "alcacovas", "barragem_de_castelo_burgoes", "rebordelo", 
    "barragem_do_divor", "barragem_do_roxo", "batalha", "campo_experimental_crato", 
    "caxarias", "colares", "comporta", "vila_nova_de_cerveira", "gondizalves", "junqueira", 
    "minas_de_jales", "proenca_a_nova", "santarem", "sao_bras_de_alportel"
]

# Optional: set this to True if you still want to keep track of the original location
add_location_column = True

# Read and union all DataFrames
dfs = []
for loc in locations:
    df = spark.read.format("delta").load(os.path.join(base_path, loc))
    if add_location_column:
        df = df.withColumn("location", lit(loc))
    dfs.append(df)

weather_df = reduce(DataFrame.unionByName, dfs)

# Show schema or a sample
display(weather_df)

## 2. Parse the timestamp

Parse the `date` field into a real timestamp (adds a `DATE` column).

In [0]:
weather_df_ts = parse_ts(weather_df, "date")

display(weather_df_ts)

Profile the combined table.

In [0]:
dbutils.data.summarize(weather_df_ts)

> **Inspection.** Look at one station's rows as a sanity check.

In [0]:
display(weather_df_ts.filter(col("location") == "colares"))

Per-station missing-value rates for the four weather variables. This is the diagnostic
that motivates the gap-filling below — it shows which stations have meaningful gaps.

In [0]:
features = ["temperatura_media_do_ar_horaria_c", "humidade_relativa_media_horaria_percent", "precipitacao_horaria_mm", "velocidade_do_vento_media_horaria_m_per_s"]

# For each column, calculate % of nulls per location
exprs = [
    (count(when(col(c).isNull(), c)) / count("*")).alias(f"{c}_null_pct")
    for c in features
]

null_stats = weather_df_ts.groupBy("location").agg(*exprs)

display(null_stats)

## 3. Fill gaps from the nearest station

Where a station is missing a reading, fill it with the value from the nearest station that
has one at the same timestamp.

How it works:
- Compute the distance between every pair of stations (Haversine, in km), using their
  coordinates.
- Rank each station's neighbours from nearest to farthest.
- For the nearest neighbour, then the second, then the third: join that neighbour's
  readings at the same timestamp and fill any value the station is still missing.

The fill uses another station's reading at the **same time**, not a different time.

In [0]:
# One row per station with coordinates
stations = (weather_df_ts
    .select("location", "latitude", "longitude")
    .dropDuplicates(["location"])
)

# Haversine distance in km
def haversine_km(lat1, lon1, lat2, lon2):
    r = F.lit(6371.0)
    dphi = F.radians(lat2 - lat1)
    dlambda = F.radians(lon2 - lon1)
    a = (
        F.pow(F.sin(dphi / 2), 2)
        + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.pow(F.sin(dlambda / 2), 2)
    )
    return r * 2 * F.asin(F.sqrt(a))

# Cross join → all pairs with distance
a = stations.alias("a")
b = stations.alias("b")

pairs = (a.crossJoin(b)
    .where(F.col("a.location") != F.col("b.location"))
    .withColumn("dist_km", haversine_km(
        F.col("a.latitude"), F.col("a.longitude"),
        F.col("b.latitude"), F.col("b.longitude")
    ))
)

# Rank all neighbours by distance (not just top 1)
w = Window.partitionBy("a.location").orderBy("dist_km")

ranked_map = (pairs
    .withColumn("rank", F.row_number().over(w))
    .select(
        F.col("a.location").alias("location"),
        F.col("b.location").alias("donor_location"),
        F.col("dist_km"),
        F.col("rank")
    )
)

# Then in the fill step, iterate through ranks until null is filled
vars_to_fill = [
    "temperatura_media_do_ar_horaria_c",
    "humidade_relativa_media_horaria_percent",
    "precipitacao_horaria_mm",
    "velocidade_do_vento_media_horaria_m_per_s"
]

df_filled = weather_df_ts

for rank in range(1, 4):  # try 3 nearest neighbours
    donor = (weather_df_ts
        .select(
            F.col("DATE"),
            F.col("location").alias("donor_location"),
            *[F.col(v).alias(f"{v}_donor") for v in vars_to_fill]
        )
    )

    ranked_at = ranked_map.filter(F.col("rank") == rank)

    df_filled = (
        df_filled
        .join(F.broadcast(ranked_at), on="location", how="left")
        .join(donor, on=["DATE", "donor_location"], how="left")
    )

    for v in vars_to_fill:
        df_filled = df_filled.withColumn(
            v, F.coalesce(F.col(v), F.col(f"{v}_donor"))
        ).drop(f"{v}_donor")

    df_filled = df_filled.drop("donor_location", "dist_km", "rank")

Profile the filled table.

In [0]:
dbutils.data.summarize(df_filled)

> **Inspection.** Compare one station before and after filling.

In [0]:
display(df_filled.filter(col("location") == "vila_nova_de_cerveira"))

In [0]:
display(weather_df_ts.filter(col("location") == "vila_nova_de_cerveira"))

## 4. Save to silver

Save the combined, filled weather table as Delta. `overwrite` plus `overwriteSchema` makes
the cell safely re-runnable.

**Target table:** `hive_metastore.silver.silver_weather`

In [0]:
target_catalog = "hive_metastore"     # change this
target_schema = "silver"
target_table = "silver_weather"  # change this

full_name = f"{target_catalog}.{target_schema}.{target_table}"


In [0]:
(
    df_filled.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_name)
)

display(spark.table(full_name).limit(20))


In [0]:
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")